In [1]:
# ============================================================
# Notebook    : 04_switchtab_pretrain.ipynb
# Description : SwitchTab pretraining extended to longitudinal
#               panel data — pairs are (IDpol, Year t) <->
#               (IDpol, Year t+1) records from the SAME
#               policyholder, NOT random samples.
#               - mutual  : time-invariant individual risk trait
#               - salient : year-specific risk deviation signal
#
#               Two-phase training:
#               Phase 1 — self-supervised pretraining (200 ep)
#               Phase 2 — supervised fine-tuning      ( 50 ep)
#
#               Speed optimizations:
#               - batch_size=1024 (RTX 4060 Ti 16GB)
#               - check_val_every_n_epoch=10 (Phase 1)
#               - check_val_every_n_epoch=5  (Phase 2)
#               - num_sanity_val_steps=0
#
#               KEY FIX: 0-indexed categorical vocab for ts3l
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install ts3l pytorch-lightning torch scikit-learn pandas


# ============================================================
# 1. Imports
# ============================================================
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import warnings
import json
import pickle
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, SequentialSampler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("high")
os.environ["TQDM_NOTEBOOK"] = "0"

DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print(f"[CHECK 1] CUDA_LAUNCH_BLOCKING : {os.environ.get('CUDA_LAUNCH_BLOCKING')}")
print(f"[CHECK 1] torch.cuda.available : {torch.cuda.is_available()}")
print(f"[CHECK 1] Device selected      : {DEVICE}")
if torch.cuda.is_available():
    print(f"[CHECK 1] GPU name            : {torch.cuda.get_device_name(0)}")
    print(f"[CHECK 1] GPU memory total    : "
          f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
    print(f"[CHECK 1] Torch version       : {torch.__version__}")


# ============================================================
# 1-B. GPU sanity test
# ============================================================
if torch.cuda.is_available():
    try:
        t = torch.randn(10, 10).cuda()
        r = (t @ t.T).sum().item()
        print(f"\n[CHECK 1-B] GPU sanity test passed. Sum: {r:.4f}")
        torch.cuda.empty_cache()
    except RuntimeError as e:
        print(f"[CHECK 1-B] !! FAILED: {e}")
        print(f"[CHECK 1-B] !! STOP — restart kernel.")
        raise


# ============================================================
# 1-C. Import ts3l after GPU health confirmed
# ============================================================
from ts3l.pl_modules import SwitchTabLightning
from ts3l.utils.switchtab_utils import (SwitchTabDataset,
                                         SwitchTabFirstPhaseCollateFN)
from ts3l.utils import TS3LDataModule
from ts3l.utils.switchtab_utils import SwitchTabConfig
from ts3l.utils.embedding_utils import FTEmbeddingConfig
from ts3l.utils.backbone_utils import TransformerBackboneConfig
from ts3l.utils.misc import get_category_cardinality
from pytorch_lightning import Trainer

import pytorch_lightning as pl
print(f"\n[CHECK 1-C] ts3l imported.")
print(f"[CHECK 1-C] pytorch_lightning : {pl.__version__}")

os.makedirs("data/switchtab", exist_ok=True)


# ============================================================
# 2. Load preprocessed multi-history data (from notebook 01)
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_features.csv")
print(f"\n[CHECK 2] df shape          : {df.shape}")
print(f"[CHECK 2] Unique IDpol count: {df['IDpol'].nunique():,}")


# ============================================================
# 3. Build adjacent-year pairs PER POLICYHOLDER
#    - Core contribution: same-IDpol, adjacent-year (t, t+1)
#      pairing instead of random sample pairing
# ============================================================
FEATURE_COLS = ["Expo", "YearGap", "Usage", "VehType", "VehPower"]
CAT_COLS     = ["Usage", "VehType", "VehPower"]
CONT_COLS    = ["Expo", "YearGap"]

df_sorted = df.sort_values(["IDpol", "Year"]).reset_index(drop=True)

pairs_x1, pairs_x2, pair_idpol, pair_year_t = [], [], [], []

for idpol, group in df_sorted.groupby("IDpol"):
    group = group.sort_values("Year").reset_index(drop=True)
    n = len(group)
    for i in range(n - 1):
        if group.loc[i + 1, "Year"] == group.loc[i, "Year"] + 1:
            pairs_x1.append(group.loc[i, FEATURE_COLS].values)
            pairs_x2.append(group.loc[i + 1, FEATURE_COLS].values)
            pair_idpol.append(idpol)
            pair_year_t.append(group.loc[i, "Year"])

X1 = pd.DataFrame(pairs_x1, columns=FEATURE_COLS)
X2 = pd.DataFrame(pairs_x2, columns=FEATURE_COLS)

print(f"\n[CHECK 3] Total adjacent-year pairs: {len(X1):,}")
print(f"[CHECK 3] X1/X2 row count match    : {len(X1) == len(X2)}")


# ============================================================
# 4. Encode categoricals to 0-INDEXED integer codes
#    - ts3l FeatureTokenizer assumes cardinality N → values 0..N-1
#    - vocabs.json starts at 1 (<PAD>=0), causing CUDA assert
#    - SwitchTab has no padding need → separate 0-indexed vocab
# ============================================================
vocabs_0indexed = {}
for col in CAT_COLS:
    categories = sorted(df_sorted[col].dropna().unique().tolist())
    vocabs_0indexed[col] = {cat: i for i, cat in enumerate(categories)}

print(f"\n[CHECK 4] 0-indexed vocab sizes:")
for col in CAT_COLS:
    print(f"  {col}: {len(vocabs_0indexed[col])} categories, "
          f"range 0-{len(vocabs_0indexed[col])-1}")

for col in CAT_COLS:
    X1[col] = X1[col].map(vocabs_0indexed[col])
    X2[col] = X2[col].map(vocabs_0indexed[col])

for col in CAT_COLS:
    n_bad = X1[col].isna().sum() + X2[col].isna().sum()
    if n_bad > 0:
        print(f"[CHECK 4] !! WARNING: {col} has {n_bad} unmapped values")

for col in CAT_COLS:
    X1[col] = X1[col].astype(int)
    X2[col] = X2[col].astype(int)

scaler = MinMaxScaler()
X1[CONT_COLS] = scaler.fit_transform(X1[CONT_COLS])
X2[CONT_COLS] = scaler.transform(X2[CONT_COLS])

print(f"[CHECK 4] X1 cat ranges (max == cardinality-1):")
for col in CAT_COLS:
    print(f"  {col}: min={X1[col].min()}, max={X1[col].max()}, "
          f"cardinality={len(vocabs_0indexed[col])}")

with open("data/switchtab/vocabs_0indexed.json", "w", encoding="utf-8") as f:
    json.dump(vocabs_0indexed, f, ensure_ascii=False, indent=2)
print("[CHECK 4] Saved: data/switchtab/vocabs_0indexed.json")


# ============================================================
# 5. Train / val / test split — IDpol-level (from notebook 02)
# ============================================================
with open("data/sequences/train_sequences.pkl", "rb") as f:
    train_seqs = pickle.load(f)
with open("data/sequences/val_sequences.pkl", "rb") as f:
    val_seqs = pickle.load(f)
with open("data/sequences/test_sequences.pkl", "rb") as f:
    test_seqs = pickle.load(f)

train_idpols = set(s["IDpol"] for s in train_seqs)
val_idpols   = set(s["IDpol"] for s in val_seqs)
test_idpols  = set(s["IDpol"] for s in test_seqs)

print(f"\n[CHECK 5] IDpol overlap (all 0): "
      f"tr∩val={len(train_idpols & val_idpols)}, "
      f"tr∩te={len(train_idpols & test_idpols)}, "
      f"val∩te={len(val_idpols & test_idpols)}")

pair_idpol_arr = np.array(pair_idpol)
train_mask = np.isin(pair_idpol_arr, list(train_idpols))
val_mask   = np.isin(pair_idpol_arr, list(val_idpols))
test_mask  = np.isin(pair_idpol_arr, list(test_idpols))

X1_train, X2_train = (X1[train_mask].reset_index(drop=True),
                       X2[train_mask].reset_index(drop=True))
X1_val,   X2_val   = (X1[val_mask].reset_index(drop=True),
                       X2[val_mask].reset_index(drop=True))
X1_test,  X2_test  = (X1[test_mask].reset_index(drop=True),
                       X2[test_mask].reset_index(drop=True))

# Labels for Phase 2 supervised fine-tuning
label_lookup = {(row["IDpol"], row["Year"]): row["Label"]
                for _, row in df.iterrows()}

def get_pair_labels(pair_idpol_arr, pair_year_t, mask, label_lookup):
    labels = []
    idpols = pair_idpol_arr[mask]
    years  = np.array(pair_year_t)[mask]
    for idp, yr in zip(idpols, years):
        labels.append(label_lookup.get((idp, yr), 0))
    return np.array(labels, dtype=np.int64)

y_train = get_pair_labels(
    pair_idpol_arr, pair_year_t, train_mask, label_lookup)
y_val   = get_pair_labels(
    pair_idpol_arr, pair_year_t, val_mask,   label_lookup)
y_test  = get_pair_labels(
    pair_idpol_arr, pair_year_t, test_mask,  label_lookup)

print(f"\n[CHECK 5] Pair splits: "
      f"train={len(X1_train):,}, val={len(X1_val):,}, "
      f"test={len(X1_test):,}")
print(f"[CHECK 5] Label positive rates: "
      f"train={y_train.mean()*100:.2f}%, "
      f"val={y_val.mean()*100:.2f}%, "
      f"test={y_test.mean()*100:.2f}%")


# ============================================================
# 6. SwitchTab config and hyperparameters
#    Phase 1: 200 epochs self-supervised (≈ 2.5h on RTX 4060Ti)
#    Phase 2:  50 epochs supervised      (≈ 0.5h on RTX 4060Ti)
#    Total estimated: ≈ 3 hours (overnight safe)
# ============================================================
BATCH_SIZE    = 1024
MAX_EPOCHS_P1 = 200
MAX_EPOCHS_P2 = 50

embedding_config = FTEmbeddingConfig(
    input_dim=len(FEATURE_COLS),
    emb_dim=64,
    cont_nums=len(CONT_COLS),
    cat_cardinality=get_category_cardinality(X1_train, CAT_COLS),
    required_token_dim=2,
)
backbone_config = TransformerBackboneConfig(
    d_model=embedding_config.emb_dim,
    encoder_depth=2,
    n_head=2,
    hidden_dim=256,
)
config = SwitchTabConfig(
    task="classification",
    loss_fn="CrossEntropyLoss",
    metric="accuracy_score",
    metric_hparams={},
    embedding_config=embedding_config,
    backbone_config=backbone_config,
    output_dim=2,
    u_label=-1,
    optim="RMSprop",
    optim_hparams={"lr": 0.0003},
)

print(f"\n[CHECK 6] Config:")
print(f"  emb_dim         : {embedding_config.emb_dim}")
print(f"  cat_cardinality : {get_category_cardinality(X1_train, CAT_COLS)}")
print(f"  u_label         : {config.u_label}")
print(f"  output_dim      : {config.output_dim}")
print(f"  batch_size      : {BATCH_SIZE}")
print(f"  Phase 1 epochs  : {MAX_EPOCHS_P1}")
print(f"  Phase 2 epochs  : {MAX_EPOCHS_P2}")
print(f"  steps/epoch est : ~{len(X1_train) // BATCH_SIZE}")

pl_model = SwitchTabLightning(config)
print(f"\n[CHECK 6] Model created. "
      f"Params: {sum(p.numel() for p in pl_model.parameters()):,}")

# Phase 1 uses dummy labels (-1 = unlabeled, self-supervised only)
dummy_y_train = np.full(len(X1_train), -1)
dummy_y_val   = np.full(len(X1_val),   -1)
print(f"[CHECK 6] dummy_y unique (should be [-1]): "
      f"{np.unique(dummy_y_train)}")


# ============================================================
# 7. Phase 1 datasets and datamodule
# ============================================================
train_ds_p1 = SwitchTabDataset(
    X=X1_train,
    unlabeled_data=X2_train,
    Y=dummy_y_train,
    config=config,
    continuous_cols=CONT_COLS,
    category_cols=CAT_COLS,
)
valid_ds_p1 = SwitchTabDataset(
    X=X1_val,
    unlabeled_data=X2_val,
    Y=dummy_y_val,
    config=config,
    continuous_cols=CONT_COLS,
    category_cols=CAT_COLS,
)

datamodule_p1 = TS3LDataModule(
    train_ds_p1, valid_ds_p1,
    batch_size=BATCH_SIZE,
    train_sampler="random",
    train_collate_fn=SwitchTabFirstPhaseCollateFN(),
    valid_collate_fn=SwitchTabFirstPhaseCollateFN(),
    n_jobs=0,
)

print(f"\n[CHECK 7] Phase 1 datasets: "
      f"train={len(train_ds_p1):,}, val={len(valid_ds_p1):,}")

# Batch sanity check
datamodule_p1.setup(stage="fit")
sample = next(iter(datamodule_p1.train_dataloader()))
print(f"[CHECK 7] Batch sanity — "
      f"batch[0]:{sample[0].shape}, "
      f"batch[1]:{sample[1].shape}, "
      f"batch[2]:{sample[2].shape}")
print(f"[CHECK 7] Phase 1 sanity check PASSED.")


# ============================================================
# 8. Phase 1 — Self-supervised pretraining
#    - check_val_every_n_epoch=10: validation every 10 epochs
#      instead of every epoch → ~50% speed improvement
#    - num_sanity_val_steps=0: skip initial sanity check
# ============================================================
print(f"\n[CHECK 8] Starting Phase 1 "
      f"({MAX_EPOCHS_P1} epochs, batch={BATCH_SIZE})...")
print(f"[CHECK 8] Validation every 10 epochs → speed optimized")
print(f"[CHECK 8] Estimated time: ~{MAX_EPOCHS_P1 * 7 // 60}h "
      f"{MAX_EPOCHS_P1 * 7 % 60}min")

trainer_p1 = Trainer(
    accelerator=DEVICE,
    max_epochs=MAX_EPOCHS_P1,
    num_sanity_val_steps=0,          # skip initial sanity check
    check_val_every_n_epoch=10,      # validate every 10 epochs only
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)
trainer_p1.fit(pl_model, datamodule_p1)

print(f"\n[CHECK 8] Phase 1 complete.")

torch.save(pl_model.state_dict(),
           "data/switchtab/switchtab_phase1.pt")
print(f"[CHECK 8] Saved: data/switchtab/switchtab_phase1.pt")


# ============================================================
# 9. Phase 2 — Supervised fine-tuning
#    - set_second_phase(): activates classification head
#    - Real labels (y_train, y_val) used
#    - Optimizer → Adam lr=0.001 (original paper Phase 2)
#    - train_sampler="weighted": corrects class imbalance
#    - check_val_every_n_epoch=5: validate every 5 epochs
# ============================================================
print(f"\n[CHECK 9] Switching to Phase 2 "
      f"({MAX_EPOCHS_P2} epochs, supervised)...")

pl_model.set_second_phase()
pl_model.optim         = torch.optim.Adam
pl_model.optim_hparams = {"lr": 0.001}

print(f"[CHECK 9] Optimizer: Adam lr=0.001")

train_ds_p2 = SwitchTabDataset(
    X=X1_train,
    Y=y_train,
    continuous_cols=CONT_COLS,
    category_cols=CAT_COLS,
    is_second_phase=True,
)
valid_ds_p2 = SwitchTabDataset(
    X=X1_val,
    Y=y_val,
    continuous_cols=CONT_COLS,
    category_cols=CAT_COLS,
    is_second_phase=True,
)

datamodule_p2 = TS3LDataModule(
    train_ds_p2, valid_ds_p2,
    batch_size=BATCH_SIZE,
    train_sampler="weighted",
    n_jobs=0,
)

print(f"[CHECK 9] Phase 2 datasets: "
      f"train={len(train_ds_p2):,}, val={len(valid_ds_p2):,}")

trainer_p2 = Trainer(
    accelerator=DEVICE,
    max_epochs=MAX_EPOCHS_P2,
    num_sanity_val_steps=0,
    check_val_every_n_epoch=5,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)
trainer_p2.fit(pl_model, datamodule_p2)

print(f"\n[CHECK 9] Phase 2 complete.")

torch.save(pl_model.state_dict(),
           "data/switchtab/switchtab_phase2.pt")
print(f"[CHECK 9] Saved: data/switchtab/switchtab_phase2.pt")


# ============================================================
# 10. Evaluate SwitchTab as standalone classifier on test set
#     Result added as 7th model row in notebook 06 comparison
# ============================================================
print(f"\n[CHECK 10] Evaluating SwitchTab standalone on test set...")

test_ds = SwitchTabDataset(
    X=X1_test,
    continuous_cols=CONT_COLS,
    category_cols=CAT_COLS,
    is_second_phase=True,
)
test_dl = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    sampler=SequentialSampler(test_ds)
)

preds = trainer_p2.predict(pl_model, test_dl)
preds = F.softmax(
    torch.concat([o.cpu() for o in preds]).squeeze(), dim=1
)
y_pred_prob = preds[:, 1].numpy()
y_pred_cls  = preds.argmax(1).numpy()

y_test_aligned = y_test[:len(y_pred_prob)]

auc   = roc_auc_score(y_test_aligned, y_pred_prob)
f1_05 = f1_score(y_test_aligned, y_pred_cls)

precisions, recalls, thresholds = precision_recall_curve(
    y_test_aligned, y_pred_prob)
f1s      = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1s[:-1])
best_f1  = f1s[best_idx]
best_thr = thresholds[best_idx]

print(f"\n[CHECK 10] ===== SwitchTab Standalone Test Results =====")
print(f"  Total test pairs  : {len(y_test_aligned):,}")
print(f"  Positive rate     : {y_test_aligned.mean()*100:.2f}%")
print(f"  AUC-ROC           : {auc:.4f}")
print(f"  F1 @ thr=0.5      : {f1_05:.4f}")
print(f"  Best threshold    : {best_thr:.3f}")
print(f"  Best F1           : {best_f1:.4f}")

np.savez(
    "data/switchtab/switchtab_test_predictions.npz",
    probs=y_pred_prob,
    labels=y_test_aligned,
)
print(f"[CHECK 10] Saved: data/switchtab/switchtab_test_predictions.npz")


# ============================================================
# 11. Extract mutual / salient embeddings for the FULL dataset
#     Used by notebook 05 as Transformer input
# ============================================================
df_full = df_sorted.copy()
for col in CAT_COLS:
    df_full[col] = df_full[col].map(vocabs_0indexed[col]).astype(int)
df_full[CONT_COLS] = scaler.transform(df_full[CONT_COLS])

print(f"\n[CHECK 11] Extracting embeddings "
      f"({df_full.shape[0]:,} rows)...")

full_ds = SwitchTabDataset(
    X=df_full[FEATURE_COLS],
    continuous_cols=CONT_COLS,
    category_cols=CAT_COLS,
    is_second_phase=True,
)
full_dl = DataLoader(
    full_ds, batch_size=1024, shuffle=False,
    sampler=SequentialSampler(full_ds)
)

pl_model.eval()
mutual_list, salient_list = [], []

with torch.no_grad():
    for batch_idx, batch in enumerate(full_dl):
        x = batch[0] if isinstance(batch, (list, tuple)) else batch

        z       = pl_model.model.embedding_module(x)
        z       = pl_model.model.encoder(z)
        mutual  = pl_model.model.projector_m(z)
        salient = pl_model.model.projector_s(z)

        mutual_list.append(mutual.cpu().numpy())
        salient_list.append(salient.cpu().numpy())

        if batch_idx == 0:
            print(f"[CHECK 11] First batch — "
                  f"z:{z.shape}, "
                  f"mutual:{mutual.shape}, "
                  f"salient:{salient.shape}")

mutual_emb  = np.concatenate(mutual_list,  axis=0)
salient_emb = np.concatenate(salient_list, axis=0)

print(f"[CHECK 11] Mutual  shape : {mutual_emb.shape}")
print(f"[CHECK 11] Salient shape : {salient_emb.shape}")
print(f"[CHECK 11] NaN — "
      f"mutual:{np.isnan(mutual_emb).any()}, "
      f"salient:{np.isnan(salient_emb).any()}")

np.savez(
    "data/switchtab/embeddings.npz",
    mutual=mutual_emb,
    salient=salient_emb,
    IDpol=df_full["IDpol"].values,
    Year=df_full["Year"].values,
)
print("[CHECK 11] Saved: data/switchtab/embeddings.npz")


# ============================================================
# 12. Summary
# ============================================================
print(f"\n===== SwitchTab Two-Phase Training Summary =====")
print(f"Pairing strategy     : same-IDpol adjacent-year (t, t+1)")
print(f"Train/Val/Test pairs : "
      f"{len(X1_train):,} / {len(X1_val):,} / {len(X1_test):,}")
print(f"Phase 1 epochs       : {MAX_EPOCHS_P1} (self-supervised, "
      f"val every 10 ep)")
print(f"Phase 2 epochs       : {MAX_EPOCHS_P2} (supervised, "
      f"val every 5 ep)")
print(f"Batch size           : {BATCH_SIZE}")
print(f"SwitchTab AUC        : {auc:.4f}")
print(f"SwitchTab Best F1    : {best_f1:.4f}")
print(f"Mutual  embedding    : {mutual_emb.shape}")
print(f"Salient embedding    : {salient_emb.shape}")
print(f"================================================")

[CHECK 1] CUDA_LAUNCH_BLOCKING : 1
[CHECK 1] torch.cuda.available : True
[CHECK 1] Device selected      : gpu
[CHECK 1] GPU name            : NVIDIA GeForce RTX 4060 Ti
[CHECK 1] GPU memory total    : 16.0 GB
[CHECK 1] Torch version       : 2.6.0+cu124

[CHECK 1-B] GPU sanity test passed. Sum: 44.9087

[CHECK 1-C] ts3l imported.
[CHECK 1-C] pytorch_lightning : 2.6.5

[CHECK 2] df shape          : (364997, 11)
[CHECK 2] Unique IDpol count: 71,392

[CHECK 3] Total adjacent-year pairs: 293,532
[CHECK 3] X1/X2 row count match    : True

[CHECK 4] 0-indexed vocab sizes:
  Usage: 18 categories, range 0-17
  VehType: 15 categories, range 0-14
  VehPower: 8 categories, range 0-7
[CHECK 4] X1 cat ranges (max == cardinality-1):
  Usage: min=0, max=17, cardinality=18
  VehType: min=0, max=14, cardinality=15
  VehPower: min=0, max=7, cardinality=8
[CHECK 4] Saved: data/switchtab/vocabs_0indexed.json

[CHECK 5] IDpol overlap (all 0): tr∩val=0, tr∩te=0, val∩te=0


Seed set to 42



[CHECK 5] Pair splits: train=205,217, val=44,281, test=44,034
[CHECK 5] Label positive rates: train=12.87%, val=12.70%, test=12.96%

[CHECK 6] Config:
  emb_dim         : 64
  cat_cardinality : [18, 15, 8]
  u_label         : -1
  output_dim      : 2
  batch_size      : 1024
  Phase 1 epochs  : 200
  Phase 2 epochs  : 50
  steps/epoch est : ~200

[CHECK 6] Model created. Params: 178,247
[CHECK 6] dummy_y unique (should be [-1]): [-1]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                   | Type             | Params | Mode  | FLOPs
----------------------------------------------------------------------------
0 | task_loss_fn           | CrossEntropyLoss | 0      | train | 0    
1 | reconstruction_loss_fn | MSELoss          | 0      | train | 0    
2 | model                  | SwitchTab        | 178 K  | train | 0    
----------------------------------------------------------------------------
178 K     Trainable params
0         Non-trainable params
178 K     Total params
0.713     Total estimated model params size (MB)
40        Modules in train mode
0         Modules in eval mode
0         Total 


[CHECK 7] Phase 1 datasets: train=410,434, val=88,562
[CHECK 7] Batch sanity — batch[0]:torch.Size([2048, 5]), batch[1]:torch.Size([2048, 5]), batch[2]:torch.Size([2048])
[CHECK 7] Phase 1 sanity check PASSED.

[CHECK 8] Starting Phase 1 (200 epochs, batch=1024)...
[CHECK 8] Validation every 10 epochs → speed optimized
[CHECK 8] Estimated time: ~23h 20min


...

Validation: |                                                                                    | 0/? [00:00<…

`Trainer.fit` stopped: `max_epochs=50` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



[CHECK 9] Phase 2 complete.
[CHECK 9] Saved: data/switchtab/switchtab_phase2.pt

[CHECK 10] Evaluating SwitchTab standalone on test set...


Predicting: |                                                                                    | 0/? [00:00<…


[CHECK 10] ===== SwitchTab Standalone Test Results =====
  Total test pairs  : 44,034
  Positive rate     : 12.96%
  AUC-ROC           : 0.7668
  F1 @ thr=0.5      : 0.3570
  Best threshold    : 0.669
  Best F1           : 0.3822
[CHECK 10] Saved: data/switchtab/switchtab_test_predictions.npz

[CHECK 11] Extracting embeddings (364,997 rows)...
[CHECK 11] First batch — z:torch.Size([1024, 64]), mutual:torch.Size([1024, 64]), salient:torch.Size([1024, 64])
[CHECK 11] Mutual  shape : (364997, 64)
[CHECK 11] Salient shape : (364997, 64)
[CHECK 11] NaN — mutual:False, salient:False
[CHECK 11] Saved: data/switchtab/embeddings.npz

===== SwitchTab Two-Phase Training Summary =====
Pairing strategy     : same-IDpol adjacent-year (t, t+1)
Train/Val/Test pairs : 205,217 / 44,281 / 44,034
Phase 1 epochs       : 200 (self-supervised, val every 10 ep)
Phase 2 epochs       : 50 (supervised, val every 5 ep)
Batch size           : 1024
SwitchTab AUC        : 0.7668
SwitchTab Best F1    : 0.3822
Mutual